# ============================================
# 11_appliances_energy_prediction_downloader.ipynb
# UCI Appliances Energy Prediction Dataset
# ============================================

## Dataset Title
UCI Appliances Energy Prediction Dataset

## Source URL
Official dataset page:  
https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction

Direct download URL:  
https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip

## Purpose of This Dataset
This dataset is used as a real multivariate energy prediction dataset in the PhD workflow.

It is especially useful for:
- multivariate forecasting
- feature-fusion experiments
- appliance energy prediction
- weather-aware modeling
- robustness and edge-oriented testing

## Why We Need This Dataset
We need this dataset because it provides a clean real-world multivariate energy dataset with indoor measurements and weather-linked features.

According to the official UCI dataset page:
- it is a multivariate time-series dataset
- it has 19,735 instances
- it has 28 features
- it covers about 4.5 months
- it is sampled every 10 minutes
- it includes appliance energy, lighting, temperature, humidity, and weather variables
- the main downloadable file is `energydata_complete.csv` (11.4 MB)

This makes it highly suitable for:
- Notebook 3: NEO-Forecast
- Notebook 7: NEO-Robust
- Notebook 10: NEO-Edge
- Notebook 11: multivariate full-system support
- Notebook 12: reproducibility and benchmarking

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download the dataset ZIP from the official UCI source
4. Extract the ZIP
5. Identify usable files
6. Save a dataset manifest
7. Create a clean ZIP bundle
8. Save everything in Google Drive
9. Create a final package ZIP
10. Download that final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 3: NEO-Forecast
- Notebook 7: NEO-Robust
- Notebook 10: NEO-Edge
- Notebook 11: Full System Simulation support
- Notebook 12: Reproducibility and benchmarking

## Notes
The official UCI page states that this dataset is a multivariate time-series dataset with 19,735 instances and 28 features, sampled every 10 minutes, and that the main downloadable file is `energydata_complete.csv`.

In [1]:
# ============================================
# 11_appliances_energy_prediction_downloader.ipynb
# FULL FIXED SINGLE CELL
# ============================================

import os
import json
import shutil
import zipfile
from pathlib import Path
import requests
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "UCI Appliances Energy Prediction Dataset"
SOURCE_PAGE = "https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction"
DOWNLOAD_URL = "https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "11_appliances_energy_prediction"
raw_dir = dataset_root / "raw"
extracted_dir = dataset_root / "extracted"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, extracted_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", extracted_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=300, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download dataset ZIP
# -----------------------------
download_path = raw_dir / "appliances_energy_prediction.zip"

if not download_path.exists():
    print("\nDownloading dataset ZIP...")
    download_file(DOWNLOAD_URL, download_path)
    print("Download complete:", download_path)
else:
    print("\nUsing cached file:", download_path)

if not download_path.exists():
    raise FileNotFoundError(f"Dataset ZIP not found: {download_path}")

print("ZIP size:", round(download_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 6. Extract ZIP
# -----------------------------
print("\nExtracting ZIP...")
with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extracted_dir)

print("Extraction complete:", extracted_dir)

# -----------------------------
# 7. Collect usable files
# -----------------------------
all_files = sorted([p for p in extracted_dir.rglob("*") if p.is_file()])

usable_files = []
for p in all_files:
    rel = str(p.relative_to(extracted_dir))
    if "__MACOSX" in rel or p.name.startswith("._"):
        continue
    if p.suffix.lower() in [".csv", ".txt", ".json", ".xlsx", ".xls", ".md", ".data"]:
        usable_files.append(p)

print("\nTotal extracted files:", len(all_files))
print("Usable files:", len(usable_files))
for fp in usable_files[:20]:
    print(" -", fp)

# -----------------------------
# 8. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "downloaded_file": str(download_path),
    "extracted_dir": str(extracted_dir),
    "all_files": [str(p) for p in all_files],
    "usable_files": [str(p) for p in usable_files],
    "used_in_next_notebooks": [
        "Notebook 3: NEO-Forecast",
        "Notebook 7: NEO-Robust",
        "Notebook 10: NEO-Edge",
        "Notebook 11: Full System Simulation support",
        "Notebook 12: Reproducibility and benchmarking"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 9. Create clean bundle ZIP
# -----------------------------
clean_bundle_zip_path = bundle_dir / "appliances_energy_prediction_CLEAN_bundle.zip"

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in usable_files:
        if not fp.exists():
            continue
        arcname = str(fp.relative_to(extracted_dir))
        zf.write(fp, arcname=arcname)

print("Created clean bundle ZIP:", clean_bundle_zip_path)
print("Clean bundle size:", round(clean_bundle_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 10. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "appliances_energy_prediction_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(download_path, arcname="raw/" + download_path.name)

    for file in extracted_dir.rglob("*"):
        if file.is_file():
            rel = str(file.relative_to(extracted_dir))
            if "__MACOSX" in rel or file.name.startswith("._"):
                continue
            zf.write(file, arcname="extracted/" + rel)

    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 11. Copy final ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/appliances_energy_prediction_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 12. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Raw ZIP              :", download_path)
print("Extracted folder     :", extracted_dir)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 13. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : UCI Appliances Energy Prediction Dataset
Source Page  : https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction
Download URL : https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip
Mounted at /content/drive

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/11_appliances_energy_prediction
 - /content/drive/MyDrive/PhD_Thesis_Datasets/11_appliances_energy_prediction/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/11_appliances_energy_prediction/extracted
 - /content/drive/MyDrive/PhD_Thesis_Datasets/11_appliances_energy_prediction/bundle

Download complete: /content/drive/MyDrive/PhD_Thesis_Datasets/11_appliances_energy_prediction/raw/appliances_energy_prediction.zip
ZIP size: 11.42 MB

Extracting ZIP...
Extraction complete: /content/drive/MyDrive/PhD_Thesis_Datasets/11_appliances_energy_prediction/extracted

Total extracted files: 1
Usable files: 1
 - /content/drive/MyDrive

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>